In [143]:
import requests
from bs4 import BeautifulSoup

import re
import pandas as pd
from rapidfuzz import fuzz

import asyncio

# uv add playwright
# uv run playwright install
# uv run playwright install-deps
from playwright.async_api import async_playwright 

## Scrape and Validation Functions

In [284]:
def scrape_autokinito(base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):   
    # Set target page
    url = f"{base}/antiprosopies/"

    # Get the page's HTML
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    return soup


def get_aftokinito_cars(soup, base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):
    # Car brands and models are located under the "make/" URL path
    make_links = []

    # Find all <a> tags where the href contains the /make/ path
    for a in soup.select("a[href*='/make/']"):
        make_links.append(base + a["href"])

    cars = []

    # Loop through each brand link
    for link in set(make_links):
        r = requests.get(link, headers=HEADERS)
        s = BeautifulSoup(r.text, "html.parser")

        # Extract brand name
        brand = link.rstrip("/").split("/")[-1].upper()

        # Extract models
        for m in s.select("h2, h3"):
            model = m.get_text(strip=True)
            if model:
                cars.append((brand, model))
    
    return cars
    

def validate_aftokinito_cars(cars):
    valid_models = dict()

    for car in cars:
        brand = car[0].strip() #.lower().strip()
        model = car[1].strip() #.lower().strip()
        
        if brand not in valid_models:
            valid_models[brand] = []
        
        # In aftokinito's site, each model starts with its brand, but the way brand is stored in model may slightly differ
        model_prefix = model[:len(brand)]
        similarity = fuzz.ratio(brand, model_prefix)
        
        if similarity >= 70:
            valid_models[brand].append(model)

    return valid_models


def convert_aftokinito_to_pandas_and_clean(cars):
    rows = [(brand, model) for brand, models in cars.items() for model in models]
    df = pd.DataFrame(rows, columns=['Brand', 'Model'])

    df = df.drop_duplicates()

    # Remove rows where 'Model' contains Greek characters: these are adverts
    df = df[~df['Model'].str.contains(r'[α-ωΑ-Ω]', regex=True)]

    # Remove brand prefix from Model for every brand 
    df["Model"] = (
        df["Model"]
        .str.replace(
            r'^(' + '|'.join(df["Brand"].unique()) + r')\s+',
            '',
            regex=True
        )
    )
    df.loc[df["Brand"] == "ALFA-ROMEO", "Model"] = df.loc[df["Brand"] == "ALFA-ROMEO", "Model"].str.replace("ALFA ROMEO ", "",  regex=False)
    df.loc[df["Brand"] == "MG", "Model"] = df.loc[df["Brand"] == "MG", "Model"].str.replace("MG", "", regex=False)

    df["Source_URL"] = "https://autokinito.com.cy/antiprosopies/"

    return df.reset_index(drop=True)

In [272]:
async def scrape_bazaraki(pages=1):
    base_url = "https://www.bazaraki.com/car-motorbikes-boats-and-parts/cars-trucks-and-vans/"
    soups = []
    
    # Blocked by Cloudflare: https://scrapeops.io/web-scraping-playbook/how-to-bypass-cloudflare/ 
    # Bypass using playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # important for Cloudflare
            args=["--disable-blink-features=AutomationControlled"]
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 900},
            locale="en-US"
        )

        page = await context.new_page()

        for i in range(1, pages + 1):
            url = f"{base_url}?page={i}"

            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # wait for page structure 
            await page.wait_for_selector("body")

            await page.wait_for_timeout(3000)  # allow JS rendering

            html = await page.content()
            soups.append(BeautifulSoup(html, "html.parser"))

        await browser.close()

    return soups


def get_bazaraki_cars(soups):
    cars = []
    for soup in soups:

        # Ads in div.advert; brand + model in <a.advert__content-title>
        for advert in soup.select("div.advert"):

            title_tag = advert.select_one("a.advert__content-title")
            if not title_tag:
                continue

            title = title_tag.get_text(strip=True)
            cars.append(title)

    return cars


def convert_bazaraki_to_pandas_and_clean(cars):
    # Remove engine sizes and year
    cars = [
        re.sub(r"\s\d{4}$", "",
        re.sub(r"\s\d,\dL", "", car))
        for car in cars
    ]

    brand_model_split = [car.split() for car in cars]
    df = pd.DataFrame(brand_model_split) # columns are numbered 0, 1, 2…

    df = df.drop_duplicates()
    df = df[df[0] != "Cars"]

    # In some cars, parts of the brand appear in model because the brand is split across multiple words.
    # For example, "Land Rover" gets "Land" as the brand and "Rover" as the model.
    # This is to verify that brands and models are captured correctly
    # print(df[0].unique())
    idx = df.index[
        ((df[0] == "Land") & (df[1] == "Rover")) |
        ((df[0] == "Alfa") & (df[1] == "Romeo")) |
        ((df[0] == "Rolls") & (df[1] == "Royce")) |
        ((df[0] == "Aston") & (df[1] == "Martin"))
    ]

    # Combine columns 0 and 1 into 0
    df.loc[idx, 0] = df.loc[idx, 0] + ' ' + df.loc[idx, 1]

    # Combine remaining columns 2,3,4 into 1, ignoring NaNs
    df.loc[idx, 1] = df.loc[idx, [2, 3, 4]].fillna("").agg(' '.join, axis=1).str.strip()

    # Clear original 2,3,4 columns
    df.loc[idx, [2, 3, 4]] = pd.NA

    # Combine 1,2,3,4 columns into 1 for the remaining records
    df.loc[:, 1] =  df.loc[:, [1,2,3,4]].fillna("").agg(' '.join, axis=1).str.strip()

    # Mark 0 as Brand and 1 as Model
    df = df[[0,1]].rename(columns={0: 'Brand', 1: 'Model'})
    df["Source_URL"] = "https://www.bazaraki.com/car-motorbikes-boats-and-parts/cars-trucks-and-vans/"

    return df.reset_index(drop=True)

## Scrape and Clean 

In [316]:
# aftokinito_soup = scrape_autokinito()
# aftokinito_cars = get_aftokinito_cars(aftokinito_soup)
# valid_aftokinito_cars = validate_aftokinito_cars(aftokinito_cars)
# aftokinito = convert_aftokinito_to_pandas_and_clean(valid_aftokinito_cars)

# bazaraki_soups = await scrape_bazaraki(175)
# bazaraki_cars = get_bazaraki_cars(bazaraki_soups)
# bazaraki = convert_bazaraki_to_pandas_and_clean(bazaraki_cars)

aftokinito_bazaraki = pd.concat([aftokinito, bazaraki], axis=0, ignore_index=True)

## Format Brand Based on OEM (Original Equipment Manufacturer)

In [321]:
# ChatGPT generated
brand_mapping = {
    'CITROEN': 'Citroën',
    'ALFA-ROMEO': 'Alfa Romeo',
    'LAND ROVER': 'Land Rover',
    'VOLKSWAGEN': 'Volkswagen',
    'MERCEDES': 'Mercedes-Benz',
    'OPEL,': 'Opel',
    'JEEP': 'Jeep',
    'BMW': 'BMW',
    'FORD': 'Ford',
    'NISSAN': 'Nissan',
    'MAZDA': 'Mazda',
    'MASERATI': 'Maserati',
    'TESLA': 'Tesla',
    'PORSCHE': 'Porsche',
    'HONDA': 'Honda',
    'FIAT': 'Fiat',
    'KIA': 'Kia',
    'AUDI': 'Audi',
    'RENAULT': 'Renault',
    'ISUZU': 'Isuzu',
    'LAMBORGHINI': 'Lamborghini',
    'BENTLEY': 'Bentley',
    'LEXUS': 'Lexus',
    'SUBARU': 'Subaru',
    'SUZUKI': 'Suzuki',
    'DACIA': 'Dacia',
    'CHEVROLET': 'Chevrolet',
    'MITSUBISHI': 'Mitsubishi',
    'INFINITI': 'Infiniti',
    'PEUGEOT': 'Peugeot',
    'CADILLAC': 'Cadillac',
    'ROLLS ROYCE': 'Rolls Royce',
    'SMART': 'Smart',
    'SAAB': 'Saab',
    'FERRARI': 'Ferrari',
    'ASTON MARTIN': 'Aston Martin',
    'HYUNDAI': 'Hyundai',
    'MCLAREN': 'McLaren',
    'CHRYSLER': 'Chrysler',
    'LOTUS': 'Lotus',
    'MAHINDRA': 'Mahindra',
    'DAIHATSU': 'Daihatsu',
    'SSANGYONG': 'SsangYong',
    'MG': 'MG',
    'GENESIS': 'Genesis',
    'HUMMER': 'Hummer',
    'LINCOLN': 'Lincoln',
    'DAEWOO': 'Daewoo',
    'BAIC': 'BAIC',
    'TOYOTA': 'Toyota',
    'SKODA': 'Skoda',
    'VOLVO': 'Volvo',
    'OPEL': 'Opel',
    'MINI': 'MINI',
    'CUPRA': 'CUPRA',
    'SEAT': 'SEAT'
}

# Apply mapping (case-insensitive)
aftokinito_bazaraki['Brand'] = aftokinito_bazaraki['Brand'].apply(lambda x: brand_mapping.get(x.upper(), x))
aftokinito_bazaraki["Brand"].unique()

<StringArray>
[          'BYD',       'Citroën',           'Kia',         'Mazda',
        'Suzuki',       'Renault', 'Mercedes-Benz',         'Honda',
          'Audi',         'Dacia',         'CUPRA',        'Toyota',
       'Hyundai',         'Isuzu',         'Skoda',          'SEAT',
       'Peugeot',         'Lexus',    'Volkswagen',      'Maserati',
        'Subaru',    'Alfa Romeo',            'MG',          'BAIC',
          'Jeep',         'Volvo',          'Opel',           'BMW',
          'Fiat',       'Porsche',          'Ford',        'Nissan',
       'Bentley',    'Land Rover',         'Tesla',        'Jaguar',
         'XPeng',          'MINI',   'Lamborghini',     'Chevrolet',
    'Mitsubishi',      'Infiniti',      'Cadillac',   'Rolls Royce',
         'Dodge',         'Smart',          'Saab',       'Ferrari',
  'Aston Martin',       'McLaren',      'Chrysler',         'Lotus',
      'Mahindra',      'Daihatsu',     'SsangYong',       'Genesis',
        'Hummer',   

## Format Model Based on OEM

In [322]:
# ChatGpt generated
model_mapping = {
    # Hyphen / formatting fixes
    "C-RV Hybrid": "CR-V Hybrid",
    "HS Plug in Hybrid": "HS Plug-in Hybrid",
    "NX 450h Plug in Hybrid": "NX 450h Plug-in Hybrid",
    "Prius Plug in Hybrid": "Prius Plug-in Hybrid",
    "Rav4 Plug in Hybrid": "Rav4 Plug-in Hybrid",
    "A-Class PHEV": "A-Class PHEV",  # already correct, can skip
    "C-Class Saloon PHEV": "C-Class Saloon PHEV",
    "E-Class Saloon PHEV": "E-Class Saloon PHEV",
    "GLE Coupe PHEV": "GLE Coupe PHEV",
    "GLE SUV PHEV": "GLE SUV PHEV",
    "GLC SUV PHEV": "GLC SUV PHEV",
    "GLC Coupe PHEV": "GLC Coupe PHEV",
    "GLC AMG Coupe": "GLC AMG Coupe",  # check if needed
    "CLA Coupe EQ": "CLA Coupe EQ",
    
    # EV naming normalization
    "ID.3 Electric": "ID.3 EV",
    "ID.4 Electric": "ID.4 EV",
    "ID.5 EV": "ID.5 EV",  # correct
    "ID Buzz Cargo EV": "ID. Buzz Cargo EV",
    "ID Buzz People EV": "ID. Buzz People EV",
    "e-Transit Van EV": "e-Transit Van EV",
    "Townstar Van EV": "Townstar Van EV",
    "Atto 3 Electric": "Atto 3 EV",
    "Sealion 7 Electric": "Sealion 7 EV",
    "Mustang Mach-E Electric": "Mustang Mach-E EV",
    "Model 3 Performance Electric": "Model 3 Performance EV",
    "Model Y Electric": "Model Y EV",
    "Model S Electric": "Model S EV",
    "Model X Electric": "Model X EV",
    "iX2 Electric": "iX2 EV",
    "iX1 Electric": "iX1 EV",
    "iX Electric": "iX EV",
    "i4 Electric": "i4 EV",
    "EX30 EV": "EX30 EV",  # correct, can skip
    "EX40 EV": "EX40 EV",
    "EX90 EV": "EX90 EV",
    "EC40 EV": "EC40 EV",
    "RZ Electric": "RZ 300e EV",
    "UX 300e EV": "UX 300e EV",  # correct
    "NX 450h Plug in Hybrid": "NX 450h Plug-in Hybrid",
    
    # Other minor OEM formatting
    "ID Buzz Cargo EV": "ID. Buzz Cargo EV",
    "ID Buzz People EV": "ID. Buzz People EV",
    "CLE Coupe PHEV": "CLE Coupe PHEV",
    "E-Class AMG Saloon": "E-Class AMG Saloon",
    "C-Class AMG Saloon": "C-Class AMG Saloon",
    "Maybach GLS": "Maybach GLS",
    "A-Class AMG": "A-Class AMG",
    "G-Class EQ EV": "G-Class EQ EV",
    "RZ 300e EV": "RZ 300e EV",
    "CLE AMG Cabrio": "CLE AMG Cabrio",
    "CLA-Class AMG": "CLA-Class AMG",
    "C-Class AMG": "C-Class AMG",
    "EQE AMG Saloon": "EQE AMG Saloon",
    "EQE AMG SUV": "EQE AMG SUV",
    "EQS AMG Coupe": "EQS AMG Coupe",
    "EQC Electric": "EQC Electric",
    "EQS Electric": "EQS Electric",
    "e-tron GT EV": "e-tron GT EV",
    "RS e-tron GT EV": "RS e-tron GT EV",
}

aftokinito_bazaraki["Model"] = aftokinito_bazaraki["Model"].replace(model_mapping)

## Combine With Existing Cars

In [339]:
existing_cars = pd.read_excel(r"data/MasterList_OEM_EU_CY_vFINAL_ALLbrands.xlsx")
df = pd.concat([existing_cars, aftokinito_bazaraki],  axis=0, ignore_index=True)

# Drop duplicates
mask = df.duplicated(subset=['Brand', 'Model'], keep=False) & df['In_BoC_List'].isna()
df = df[~mask]

In [351]:
# For each brand, find duplicated models based on similarity scores
similarity_threshold = 95

duplicates = []

for brand, group in df.groupby("Brand"):
    models = group["Model"].tolist()

    # Loop over all models with their index
    for i, model1 in enumerate(models):

        # Loop over all models again with their index
        for j, model2 in enumerate(models):

            # Only compare pairs where the second index is greater than the first
            # This avoids comparing a model with itself (i == j)
            # and avoids duplicate comparisons (e.g., model1 vs model2 and model2 vs model1)
            
            if i < j:
                # At this point, model1 and model2 are a unique pair
                similarity_score = fuzz.ratio(model1, model2)

                if similarity_score >= similarity_threshold:
                    duplicates.append({
                        "Brand": brand,
                        "Model1": model1,
                        "Model2": model2,
                        "Similarity": similarity_score
                    })


dup_df = pd.DataFrame(duplicates)
dup_df

,Brand,Model1,Model2,Similarity
0,BMW,iX3 Electric,i3 Electric,95.652174
1,Bentley,Continental GT PHEV,Continental GTC PHEV,97.435897
2,Bentley,Continental GT,Continental GTC,96.551724
3,Fiat,Abarth 500,Abarth 500e,95.238095
4,Ford,Transit Chassis Cab,Transit E-Chassis Cab,95.000000
5,Maserati,Gran Turismo,GranTurismo,95.652174
6,Peugeot,2008 Electric,208 Electric,96.000000
7,Toyota,Land Cruiserr,Land Cruiser,96.000000
8,Volvo,C40 Electric,XC40 Electric,96.000000


In [352]:
df = df[~((df["Brand"] == "Maserati") & (df["Model"] == "Gran Turismo"))]
df = df[~((df["Brand"] == "Toyota") & (df["Model"] == "Land Cruiser"))]

In [355]:
df.to_excel(r"data/MasterList_aftokinito_bazaraki.xlsx", index=False)
df[["Brand","Model","Source_URL"]]

,Brand,Model,Source_URL
0,Alfa Romeo,Giulia,NaN
1,Alfa Romeo,Stelvio,NaN
2,Alfa Romeo,Tonale,NaN
3,Audi,A1,NaN
4,Audi,A3,NaN
5,Audi,A4,NaN
6,Audi,A5,NaN
7,Audi,A6,NaN
8,Audi,A7,NaN
9,Audi,A8,NaN
